In [1]:
import pandas as pd
df = pd.read_csv('../../02_Data/processed/final_merged_projects.csv')

c:\Users\seon\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\seon\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
df.columns

Index(['projectID', 'currencySymbol', 'backersCount', 'phaseLabel',
       'enableBoardGameProperties', 'minPlayers', 'maxPlayers', 'minAge',
       'playTime', 'fundedInSeconds', 'projectTags', 'isDiscounted',
       'isFeatured', 'installmentMinPayment', 'hasLimitedStock',
       'productCanBePurchased', 'previous_campaigns_count', 'duration_days',
       'softclose', 'campaignGoal_usd_1m', 'fundsGathered_usd_1m',
       'price_usd_1m', 'campaignGoal_usd_6m', 'fundsGathered_usd_6m',
       'price_usd_6m', 'edge_density', 'saturation', 'brightness', 'contrast',
       'rewards_count', 'R1', 'G1', 'B1', 'R2', 'G2', 'B2', 'R3', 'G3', 'B3',
       'R4', 'G4', 'B4', 'ks_color_1', 'ks_color_2', 'ks_color_3',
       'ks_color_4', 'emotion_adjective_1', 'emotion_adjective_2',
       'emotion_adjective_3', 'emotion_adjective_4', 'creator_id_0',
       'creator_id_1', 'is_pledge_master_0', 'is_pledge_master_1',
       'is_backer_0', 'is_backer_1', 'is_prior_backer_0', 'is_prior_backer_1',
    

In [ ]:
drop_columns = [
    'projectID', 'backersCount', 'phaseLabel', 'isFeatured', 
    'installmentMinPayment', 'hasLimitedStock', 'productCanBePurchased', 
    'campaignGoal_usd_6m', 'fundsGathered_usd_6m', 'price_usd_6m',
    'is_backer_0', 'is_backer_1'  
]

df = df.drop(columns=drop_columns)

In [11]:
df['projectTags']

0      Exploration, Horror, Modern, Science Fiction, ...
1      History, Strategy, Wargame, Action, Collectibl...
2                             History, Strategy, Wargame
3                Card Game, Strategy, Party game, Family
4      Strategy, Resource management, Family, Worker ...
                             ...                        
478    Strategy, Wargame, Dice Game, Area Control, Co...
479                             Fantasy, Strategy, Humor
480        Dice Game, Multiplayer, Competitive, Campaign
481           Fantasy, Strategy, Terrain Building, Humor
482         Science Fiction, Area Control, Deck Building
Name: projectTags, Length: 483, dtype: object

In [ ]:

import numpy as np
from flaml import AutoML
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import re
target_col = 'fundsGathered_usd_1m'

q1 = df[target_col].quantile(0.25)
q3 = df[target_col].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

df_filtered = df[(df[target_col] >= lower_bound) & (df[target_col] <= upper_bound)].copy()
df_filtered = df_filtered[df_filtered[target_col] > 0].reset_index(drop=True)


X_full = df_filtered.drop(columns=[target_col])
y_original = df_filtered[target_col]
y_log = np.log1p(y_original)


if 'projectTags' in df_filtered.columns:
    tags_dummies = df_filtered['projectTags'].str.get_dummies(sep=', ')
    X_full = pd.concat([X_full, tags_dummies], axis=1)

X_full.columns = [re.sub(r'[ ,\{\}:"\]\[\-]', '_', col) for col in X_full.columns]


X_train, X_test, y_train, y_test = train_test_split(X_full, y_log, test_size=0.2, random_state=42)

# AutoML 세팅
automl = AutoML()
automl_settings = {
    "time_budget": 60,        
    "metric": 'mae',
    "task": 'regression',
    "estimator_list": ['lgbm', 'xgboost'], 
    "seed": 42,
}

automl.fit(X_train=X_train, y_train=y_train, **automl_settings)

# 예측 및 역변환
preds_real = np.expm1(automl.predict(X_test))
y_test_real = np.expm1(y_test)

print("AutoML 최종 결과")
print("="*60)
print(f" 알고리즘: {automl.best_estimator}")
print(f"MAE : {mean_absolute_error(y_test_real, preds_real):,.2f} 달러")
print("="*60)

In [ ]:
import pandas as pd

best_lgbm_model = automl.model.estimator


try:
    feature_names = best_lgbm_model.feature_name_
except AttributeError:
    feature_names = automl.feature_names

importances = best_lgbm_model.feature_importances_

print(f"모델 변수 개수: {len(feature_names)}개 | 중요도 : {len(importances)}개")

df_automl_imp = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)

automl_active_features = df_automl_imp[df_automl_imp['Importance'] > 0].reset_index(drop=True)


print(f"투입 변수: {len(feature_names)}개")
print(f"기여도 변수: {len(automl_active_features)}개")
print("=" * 65)
print(automl_active_features.head(20))
print("-" * 65)